<!-- dd:dd-lesson-py-0 -->

# Python you need first

*Python · `py-0`*

Work through this with the **Delta Drills** side panel open. It picks what you practise, sends you to the cell, and records how it went — you do not need to read this notebook in order.


In [ ]:
# === Delta Drills ===
# Which lesson this notebook is, for the side panel. Nothing to run.
DD_LESSON_ID = "py-0"


In [ ]:
#@title 🔧 Delta Drills checker — run me first { display-mode: "form" }
# Delta Drills — problem checker. Generated; see scripts/colab_grader.py.
#
# `dd_check(<problem id>)` runs your `solve` against the same cases the tutor
# grades with, and tells you which ones failed. It reads `solve` out of the
# notebook, so define it (run your cell) before you check.
import base64
import json
import sys
import zlib

import numpy as np

# Filled in by the generated cell that follows this source: {qid: {fn, cases}}.
_DD_TESTS = {}

# Where the ARENA digits fixture is fetched from, also filled in by that cell.
_DD_FIXTURE_URL = ""
_DD_FIXTURE_PATH = "/delta_numbers.npy"

_DD_RTOL = 1e-5
_DD_ATOL = 1e-6


def _dd_install_fixtures():
    """Make `np.load('/delta_numbers.npy')` work here the way it does in the app.

    24 of the einops drills are written against the ARENA digits image, and the
    bank refers to it by an absolute path the backend rewrites at grade time
    (`code_runner.CODE_PREAMBLE`). Nothing rewrote it in a notebook, so those
    problems could not run at all in Colab — not the checker, not the starter
    code the learner was sent there to fill in. Downloaded on first use, so the
    six notebooks that never touch it never pay for it.
    """
    import os
    import urllib.request

    original = np.load
    if getattr(original, "_dd_patched", False):
        return

    def _load(file, *args, **kwargs):
        if str(file) == _DD_FIXTURE_PATH and not os.path.exists(_DD_FIXTURE_PATH):
            if not _DD_FIXTURE_URL:
                raise FileNotFoundError(
                    "This drill needs the ARENA digits fixture and no source was "
                    "compiled into this notebook — regenerate it."
                )
            urllib.request.urlretrieve(_DD_FIXTURE_URL, _DD_FIXTURE_PATH)
        return original(file, *args, **kwargs)

    _load._dd_patched = True
    np.load = _load


def _dd_load(blob):
    """The test payload, deflated and base64'd.

    Not encryption and not pretending to be — it is one `zlib.decompress` away.
    It is compressed because the payload for a 84-problem notebook is ~80 KB of
    JSON, and out of sight because an expanded grader cell would otherwise sit
    in the notebook spelling out the expected answer to every problem below it.
    """
    return json.loads(zlib.decompress(base64.b64decode(blob)).decode("utf-8"))


def _dd_preflight_torch():
    """Import torch once, here, where a failure can still be explained.

    Every drill cell opens with `import torch as t`, so the learner meets a
    broken torch install as a traceback through torch's own internals — the one
    reported was `AttributeError: partially initialized module 'torch' has no
    attribute 'fx'` from `torch/_export/utils.py`, raised while evaluating a
    function's annotations. That message names neither the cause nor the cure,
    and it is not even the real error: it is what a LATER import sees after an
    earlier one died partway and left the half-built module in `sys.modules`.
    Python does unwind a failed import normally, but a torch that was swapped
    on disk under a running kernel (a `pip install` mid-session) or shadowed by
    a stray `torch.py` gets far enough in to be cached before it falls over.

    So: purge the wreckage and retry ONCE, which is the whole fix whenever the
    first failure was transient, and report what actually broke when it is not.
    Importing torch in this cell rather than lazily is safe now in a way the
    `_dd_tensor` comment below still guards against for the per-comparison
    path — the bank is 448/448 torch and every notebook imports it a few cells
    down, so there is no numpy-only notebook left to charge for it.

    Never raises: a checker that refuses to load over this would take the
    lesson down with the runtime.
    """

    def _purge():
        # Submodules too, and that is the whole point. Python drops only the
        # module that raised, so `torch` goes and a `torch._export` imported
        # seconds earlier STAYS — and the next `import torch` re-runs
        # `torch/__init__.py` straight back into that stale submodule, which
        # reaches for a `torch.fx` the half-built parent has not bound yet.
        # Leaving one behind reproduces the bug instead of clearing it.
        for name in [n for n in sys.modules if n == "torch" or n.startswith("torch.")]:
            del sys.modules[name]

    def _usable(mod):
        # `import torch` does NOT re-execute a module already in sys.modules,
        # so a corpse left by a failed import is imported "successfully" and
        # the error surfaces later, from the learner's own cell. Judge the
        # object, not the statement: a torch that finished has both of these.
        return hasattr(mod, "fx") and hasattr(mod, "__version__")

    cached = sys.modules.get("torch")
    if cached is not None and not _usable(cached):
        _purge()

    for attempt in (1, 2):
        try:
            import torch
            if not _usable(torch):
                raise ImportError(
                    "torch imported but is only partially initialised "
                    "(no .fx) — an earlier import in this session died partway"
                )
            return True
        except Exception as exc:
            if attempt == 1:
                _purge()
                continue
            print(
                "⚠️  This runtime cannot import PyTorch, so no drill in this "
                "notebook will run.\n"
                "    %s: %s\n"
                "    Fix: Runtime ▸ Disconnect and delete runtime, then reopen "
                "this notebook and run\n"
                "    this cell first. If it comes back, check for a file named "
                "torch.py in /content,\n"
                "    and re-run any pip install BEFORE anything imports torch."
                % (type(exc).__name__, exc)
            )
    return False


def _dd_tensor(value):
    # torch only if something already imported it. numpy-only notebooks must
    # not pay a torch import to compare two lists of ints.
    torch = sys.modules.get("torch")
    return torch is not None and isinstance(value, torch.Tensor)


def _dd_close(a, b):
    """Tolerance compare, but ONLY when a float or complex is involved.

    torch defaults to float32 where numpy defaults to float64 and honest
    answers differ in reduction order, so exact equality fails correct work.
    Integer and boolean results stay exact — an index answer (argmax, nonzero,
    searchsorted) must never be fudged by a tolerance. Returns None to mean
    "not a float comparison, use exact equality".
    """
    try:
        floaty = any(
            np.issubdtype(x.dtype, np.floating) or np.issubdtype(x.dtype, np.complexfloating)
            for x in (a, b)
        )
        if not floaty:
            return None
        if a.shape != b.shape:
            return False
        return bool(np.allclose(a, b, rtol=_DD_RTOL, atol=_DD_ATOL, equal_nan=True))
    except Exception:
        return None


def _dd_array_equal(a, b):
    close = _dd_close(a, b)
    if close is not None:
        return close
    return bool(np.array_equal(a, b))


def _dd_equal(a, b):
    if _dd_tensor(a) or _dd_tensor(b):
        try:
            a2 = a.detach().cpu().numpy() if _dd_tensor(a) else np.asarray(a)
            b2 = b.detach().cpu().numpy() if _dd_tensor(b) else np.asarray(b)
            return _dd_array_equal(a2, b2)
        except Exception:
            # dtypes numpy cannot hold (bfloat16, conj views): equal tensors
            # must not grade as unequal — ask torch itself.
            torch = sys.modules.get("torch")
            if torch is not None and isinstance(a, torch.Tensor) and isinstance(b, torch.Tensor):
                try:
                    return bool(torch.equal(a.detach().cpu().resolve_conj(),
                                            b.detach().cpu().resolve_conj()))
                except Exception:
                    return False
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        return _dd_array_equal(np.asarray(a), np.asarray(b))
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        if len(a) != len(b):
            return False
        return all(_dd_equal(x, y) for x, y in zip(a, b))
    close = _dd_close(np.asarray(a), np.asarray(b))
    if close is not None:
        return close
    return bool(a == b)


def _dd_seed():
    # The same seed before the actual-side and the expected-side setup runs, for
    # BOTH rngs: setup executes twice, so an unseeded torch.rand in a fixture
    # would hand the two sides different data and fail an honest answer.
    np.random.seed(0)
    torch = sys.modules.get("torch")
    if torch is not None:
        torch.manual_seed(0)


def _dd_show(value, limit=320):
    try:
        text = repr(value)
    except Exception as exc:
        text = "<unrepresentable: %s>" % type(exc).__name__
    text = " ".join(text.split()) if len(text) > limit else text
    if len(text) > limit:
        text = text[: limit - 1] + "…"
    return text


def dd_check(question_id, verbose=True):
    """Grade the `solve` you just defined against this problem's cases.

    Returns True when every case passes. Prints which ones did not, with the
    fixture, what was expected and what came back — a failing grade should be
    evidence you can act on, not a verdict.
    """
    qid = str(question_id)
    entry = _DD_TESTS.get(qid)
    if entry is None:
        print("No checker for problem %s in this notebook." % qid)
        return False

    # The learner's namespace, not this function's: `solve` lives in the cell
    # they ran, and in Colab that is the caller's globals.
    try:
        env = sys._getframe(1).f_globals
    except Exception:
        env = globals()

    fn_name = entry.get("fn") or "solve"
    if fn_name not in env:
        print("❌ `%s` is not defined yet — run your solution cell first." % fn_name)
        return False

    cases = entry.get("cases") or []
    failures = []
    for i, case in enumerate(cases, 1):
        # A fresh copy per case: fixtures are exec'd, and exec'ing them into the
        # notebook's own globals would quietly overwrite whatever the learner
        # named `x` two cells ago.
        ns = dict(env)
        try:
            if case.get("setup_code"):
                _dd_seed()
                exec(case["setup_code"], ns)
            actual = eval(case["call"], ns)
            expected_setup = case.get("expected_setup_code") or case.get("setup_code")
            if expected_setup:
                _dd_seed()
                exec(expected_setup, ns)
            expected = eval(case["expected_expr"], ns)
            if not _dd_equal(actual, expected):
                failures.append((i, case, _dd_show(expected), _dd_show(actual), ""))
        except Exception as exc:
            failures.append((i, case, "", "", "%s: %s" % (type(exc).__name__, exc)))

    total = len(cases)
    if not failures:
        print("✅ Problem %s — %d/%d cases passed." % (qid, total, total))
        return True

    print("❌ Problem %s — %d of %d cases failed." % (qid, len(failures), total))
    if verbose:
        for i, case, expected, actual, error in failures:
            print("\n  case %d" % i)
            if case.get("setup_code"):
                for line in case["setup_code"].strip().splitlines():
                    print("    given     %s" % line)
            print("    called    %s" % _dd_show_source(case.get("call", "")))
            if error:
                print("    raised    %s" % error)
            else:
                print("    expected  %s" % expected)
                print("    you got   %s" % actual)
    return False


def _dd_show_source(text, limit=160):
    text = " ".join(str(text).split())
    return text if len(text) <= limit else text[: limit - 1] + "…"

_DD_FIXTURE_URL = "https://raw.githubusercontent.com/AkiraTheSquid/arena-book-colab/main/ARENA_5.0/ch-1-foundations/numbers.npy"
_dd_preflight_torch()
_dd_install_fixtures()
_DD_TESTS = _dd_load(
    "eNq9Wd9v2zYQ/lcEvdABJIFHipK1tzTJsALr0/owQDAKx1GxAEZdxO6QLcj/Ph4ly+nqzzmBcV9sw0d+JD/eL949pa6ap78k"
    "T+nnL/4r3W7Wf3dplqSr5bbb+n/ap3Tb7b59/bTa3HU8oheu1+PomckSe8F/d49fu9Wuu/vkfzyw3KXPWfI6AOksycvjEJUM"
    "wiPo4wBaBlBnSX0cgMr0eeExPFNNJFOUJeb4IkwiXci2qpYqS9StAkhe4sVLdSFmrgFITSBVhJL7ozkA4/zt8tl6EmsdSaLN"
    "EqArM/IsmlK4ZVM4fB2u8LyQLvQEFhGNWk5jHbTgKAqLygOLFMmijjI3AtNJOD+vooyVCqBrdeFGhspYPTu+hLr/slNiDQMY"
    "n9ebpRRF/XUPTF1tdw9CkI8P3zoAcrvZrNXImotkTZUG7NYJL1dpADAXzs8dALBCANJoC6TnI1NVLFO1wqZes/8JI+I4C84n"
    "YGkxVk7oCr2bN+zMTUDsx8kwrUaXMvMin0Do4HGHcXuK61gTLpBP7oOI0IoJx2wjjo/IF4TQmUtxygJ43lnZn6hnbq7fIFOB"
    "ad2sHcSLMESmAY+clfzDH/8iTWi/GzQBnJUcJoCzdhBPAKSQGwxpFKDgMGQAHqin8yWJJFe415LEUSwlOCfsYXJp6sphKEt+"
    "Xa63IBp9P2JPqYmktH2hsK03FH9z1WKBrVqqJh6XEQ1/WAho5al9G3QVIk24/7bdXzFvTq349506dWhz4NvG8r2nG6w2uhe/"
    "tcPN9DmK7HQQmSEnYrEFIzgmkSF7K5+C+kKR+4lwjR9G8oI/Th+XHy6pir4kr2tGc+xdoGxHzmCvaiDFXAqz1LYGCLVwPj/a"
    "FvBpMRBXn504+1bE3f4s4gYr9DkEABqLIPP5mzjj4DJ5uUUIziC4GLEf/h8kwXTASiHdsDUdVds6HnrCBg18gg9MN2d2wxRu"
    "3Fy8jbKOOcWU6lPbZMncP3eYFVyFqkJxTuwCkFke8sSB4cacl2EbF810dACjCaQNSfIr5t/Y6DcZleQaqP4sn1DAA9bpX6jS"
    "TF/z+xP4H5LC+PedsSXEYfHIYHlms66CWYtTztwf352wGC7fOilYX1QtIF545e8HSUE53trgJAjmr8Tl2v7UA82xlSyOJSGs"
    "HF+SxVy0XwgPgSjxginchl4JoqHPTIpQcJei7h22/1ghNzK8H4aRi5Hi6M5RXM8nymuihlNOJrIIXR6K0E1ztt5aFdvekfJk"
    "YGcnJz2hnIIafMU+M690bNXKxb1jHCSb3M9QSXOCJu5FjTzR2bSKOajkRaETLb8pnUNYW8ontEXpVD/PstAWY+pX6dgo3KD0"
    "RcepinS+QRlLSaUh6ypja6sbJ0wBq8I4lEq5kbTo7hDuZf32XtoP+3D/2N0lV35VhPXh/Z8318nV5R83UkyEJJ2/vF2RsQjl"
    "8t0VS0caq7PROGMeJ1SJIYySW7C67tY7WChX1ze/f7xUwVcIuUxw2foyeadeVNsrHVvMMYWNrwPnGCW3E7o7DnZ3nJw/XZyo"
    "1Pf58fN/xLJIlQ=="
)
print("Delta Drills checker ready — 28 problems. Run dd_check(<problem number>) under any of them.")


<!-- dd:dd-kp-python-values-and-names -->

## Values and names — what = really does

`python.values-and-names`


Everything in this course is built out of two moves: producing a **value**, and
giving that value a **name** so you can talk about it again later.

A value is a thing — the number `5`, the text `"hi"`, a list of numbers. An
**expression** is anything that produces one: `2 + 3` is an expression whose
value is `5`.

`print(...)` shows you a value. It is how a program says something out loud, and
almost every cell below ends with one.


In [ ]:
print(2 + 3)
print("two" + "three")




The `=` sign is **not** a claim that two things are equal. It is an instruction:
*work out the value on the right, then attach the name on the left to it.* The
name is a label, and the value is what the label is stuck to.


In [ ]:
total = 2 + 3
print(total)




Read `total = 2 + 3` as "let `total` be 5 from now on". Once a name exists you
can use it anywhere the value would work, including in building the next one:


In [ ]:
price = 4
quantity = 3
subtotal = price * quantity
print(subtotal)




Because the right-hand side is worked out **first**, a name can be built out of
its own old value. This trips people up until they read it in the right order:


In [ ]:
n = 10
n = n + 3
print(n)




The second line does not say "n equals n plus 3", which would be nonsense. It
says: take what `n` is now (10), add 3 (13), then re-attach the name `n` to
that. The old value is simply let go.

A name holds one value at a time, but you can hand several back together by
putting them in **parentheses**, separated by commas. That is a *tuple*, and it
is how a piece of code gives more than one answer at once.


In [ ]:
pair = (1, 2)
print(pair)


> **Watch out.** - **`=` is not a comparison** — `x = 5` sets a name; `x == 5` asks a question and
  produces `True` or `False`. Almost every early error message about assignment
  is one of these two written where the other belonged.
- **A quoted name is text, not the value** — `total` is the name; `"total"` is a
  five-letter string that has nothing to do with it.
- **Re-assigning does not change the old value** — it moves the label. Anything
  else already holding that value keeps it.


Two values, named, then combined and handed back as a pair.


In [ ]:
a = 7
b = 2

total = a + b
difference = a - b

print("total is", total)
print("difference is", difference)
print("both together:", (total, difference))




Why each step:

1. `a` and `b` are names for the two inputs, so the arithmetic below reads as
   words rather than as bare numbers.
2. `total` and `difference` name the two results. Naming a result is what lets
   the next line use it without recomputing it.
3. The final `(total, difference)` is a tuple: one value that carries both
   answers, which is exactly what a function returns when it has two things to
   say.


<!-- dd:dd-q568 -->

### Problem 568 · faded — your turn

Give the sum a name, then hand the name back.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
5
```


In [ ]:
def solve(a, b):
    """Store the sum under a name, then return the name."""
    _____ = a + b
    return total


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (2, 3)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(568)


In [ ]:
#@title 💡 Solution — Problem 568
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
def solve(a, b):
    """Store the sum under a name, then return the name."""
    total = a + b
    return total


example = (2, 3)
print(solve(*example))


Two values, named, then combined and handed back as a pair.


In [ ]:
a = 7
b = 2

total = a + b
difference = a - b

print("total is", total)
print("difference is", difference)
print("both together:", (total, difference))




Why each step:

1. `a` and `b` are names for the two inputs, so the arithmetic below reads as
   words rather than as bare numbers.
2. `total` and `difference` name the two results. Naming a result is what lets
   the next line use it without recomputing it.
3. The final `(total, difference)` is a tuple: one value that carries both
   answers, which is exactly what a function returns when it has two things to
   say.


<!-- dd:dd-q569 -->

### Problem 569 · faded — your turn

Swap two values by naming each one first.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(2, 1)
```


In [ ]:
def solve(x, y):
    """Swap, via two names."""
    first = _____
    second = x
    return (first, second)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (1, 2)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(569)


In [ ]:
#@title 💡 Solution — Problem 569
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
def solve(x, y):
    """Swap, via two names."""
    first = y
    second = x
    return (first, second)


example = (1, 2)
print(solve(*example))


<!-- dd:dd-q570 -->

### Problem 570 · independent

Write a function solve(price, qty) that returns a tuple (total, doubled): `total` is price times qty, and `doubled` is that same total added to itself. Compute `total` ONCE, store it under a name, and build `doubled` out of that name rather than multiplying again.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(12, 24)
```


In [ ]:
def solve(price, qty):
    """One computation, named once, reused."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (3, 4)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(570)


In [ ]:
#@title 💡 Solution — Problem 570
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
def solve(price, qty):
    """One computation, named once, reused."""
    total = price * qty
    doubled = total + total
    return (total, doubled)


example = (3, 4)
print(solve(*example))


<!-- dd:dd-q571 -->

### Problem 571 · independent

Write a function solve(n) that adds 3 to n, then adds 3 again, RE-USING the same name `n` both times, and returns the final value. A name can be reassigned; the right-hand side is worked out first, using the OLD value.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
6
```


In [ ]:
def solve(n):
    """Rebind the same name twice."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (0,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(571)


In [ ]:
#@title 💡 Solution — Problem 571
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
def solve(n):
    """Rebind the same name twice."""
    n = n + 3
    n = n + 3
    return n


example = (0,)
print(solve(*example))


#### Common mistakes

- **"`x = x + 1` is a contradiction."** — It is an instruction, not a claim. The
  right-hand side is worked out with the OLD value, and the name is then
  re-attached to the result.
- **"Printing a value and returning it are the same thing."** — `print` shows
  something to a human; `return` hands a value back to whatever called the
  function. A drill that prints instead of returning gives the grader nothing.
- **"A name IS the value."** — A name is a label. Two names can be stuck to the
  same value, and re-labelling one leaves the other where it was.


<!-- dd:dd-kp-python-types-and-conversion -->

## The everyday types, and converting between them

`python.types-and-conversion`


Every value has a **type**, and the type decides what the value can do. Four of
them carry almost all of the early work:

| Type | What it holds | Written as |
|---|---|---|
| `int` | a whole number | `5`, `-2`, `0` |
| `float` | a number with a fractional part | `2.5`, `-0.75`, `4.0` |
| `str` | text | `"hi"`, `'42'` |
| `bool` | a yes/no answer | `True`, `False` |

`type(value)` hands back the type itself. Its `__name__` is the readable word,
which is usually what you actually want to look at:


In [ ]:
print(type(5).__name__)
print(type(2.5).__name__)
print(type("42").__name__)
print(type(True).__name__)




Notice that `42` and `"42"` are **not** the same value. One is a number; the
other is two characters that happen to look like one. The difference shows up
the moment you use `+`, because `+` does a different job for each type:


In [ ]:
print(4 + 2)
print("4" + "2")




That is not a quirk — it is the type deciding the meaning. Numbers add; text
joins end to end.

To move between the two you **convert**, by calling the type's own name as a
function. Each conversion produces a NEW value and leaves the original alone:


In [ ]:
text = "42"
number = int(text)

print(number + 8)
print(text)




`float(...)` and `str(...)` work the same way, and going number → text →
number again gets you back where you started:


In [ ]:
print(float("2.5"))
print(str(7) + "!")
print(int(str(7)))




Two conversions that look similar and are not: `int(...)` throws the fractional
part **away**, while `round(...)` goes to the nearest whole number.


In [ ]:
print(int(3.9))
print(round(3.9))
print(int(-2.7), round(-2.7))


> **Watch out.** - **`int()` truncates, it does not round** — `int(3.9)` is `3`. If you wanted
  `4`, `round` is the call you meant.
- **`"5"` is not `5`** — a string of digits stays text until something converts
  it, and comparing the two gives `False`.
- **`bool` is its own type** — `True` behaves like `1` in arithmetic, but
  `type(True).__name__` is `"bool"`, not `"int"`.


One string, read three different ways, with the type named at each step.


In [ ]:
text = "12"

as_int = int(text)
as_float = float(text)
back_to_text = str(as_int)

print(as_int, type(as_int).__name__)
print(as_float, type(as_float).__name__)
print(back_to_text, type(back_to_text).__name__)
print("the original is untouched:", text, type(text).__name__)




Why each step:

1. `int(text)` reads the digits as a whole number. The string itself is not
   changed — conversion produces a new value.
2. `float(text)` reads the same digits as a number with a fractional part. `12`
   and `12.0` are equal in value and different in type.
3. `str(as_int)` goes back the other way, which is how a number gets glued into
   a message.


<!-- dd:dd-q574 -->

### Problem 574 · faded — your turn

Report the name of a value's type.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
'int'
```


In [ ]:
def solve(value):
    """The name of the value's type."""
    return type(value)._____


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (3,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(574)


In [ ]:
#@title 💡 Solution — Problem 574
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
def solve(value):
    """The name of the value's type."""
    return type(value).__name__


example = (3,)
print(solve(*example))


One string, read three different ways, with the type named at each step.


In [ ]:
text = "12"

as_int = int(text)
as_float = float(text)
back_to_text = str(as_int)

print(as_int, type(as_int).__name__)
print(as_float, type(as_float).__name__)
print(back_to_text, type(back_to_text).__name__)
print("the original is untouched:", text, type(text).__name__)




Why each step:

1. `int(text)` reads the digits as a whole number. The string itself is not
   changed — conversion produces a new value.
2. `float(text)` reads the same digits as a number with a fractional part. `12`
   and `12.0` are equal in value and different in type.
3. `str(as_int)` goes back the other way, which is how a number gets glued into
   a message.


<!-- dd:dd-q575 -->

### Problem 575 · faded — your turn

Text in, arithmetic out — the digits have to become a number first.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
50
```


In [ ]:
def solve(digits):
    """Text in, number out."""
    return _____(digits) + 8


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ('42',)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(575)


In [ ]:
#@title 💡 Solution — Problem 575
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
def solve(digits):
    """Text in, number out."""
    return int(digits) + 8


example = ('42',)
print(solve(*example))


<!-- dd:dd-q576 -->

### Problem 576 · independent

Write a function solve(text) that takes a string of digits and returns the 3-tuple (as_int, as_float, back_to_text): the value as an int, the same value as a float, and str() of the int. Converting is a one-way trip each time — nothing changes the original string.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(7, 7.0, '7')
```


In [ ]:
def solve(text):
    """One string, three ways of reading it."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ('7',)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(576)


In [ ]:
#@title 💡 Solution — Problem 576
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
def solve(text):
    """One string, three ways of reading it."""
    as_int = int(text)
    as_float = float(text)
    return (as_int, as_float, str(as_int))


example = ('7',)
print(solve(*example))


<!-- dd:dd-q577 -->

### Problem 577 · independent

Write a function solve(x) that takes a float and returns the tuple (truncated, rounded): int(x), which throws the fraction AWAY, and round(x), which goes to the nearest whole number. Show that the two disagree.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(3, 4)
```


In [ ]:
def solve(x):
    """int() truncates; round() goes to the nearest."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (3.9,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(577)


In [ ]:
#@title 💡 Solution — Problem 577
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
def solve(x):
    """int() truncates; round() goes to the nearest."""
    return (int(x), round(x))


example = (3.9,)
print(solve(*example))


#### Common mistakes

- **"Converting changes the value."** — It produces a new one. `int(text)` hands
  back a number and leaves `text` exactly as it was.
- **"int() rounds."** — It truncates toward zero: `int(3.9)` is `3` and
  `int(-2.7)` is `-2`.
- **"If it looks like a number it is one."** — `"42"` looks like a number to a
  human and is text to Python. Only a conversion makes it arithmetic-ready.


<!-- dd:dd-kp-python-lists-and-tuples -->

## Lists and tuples — holding more than one value

`python.lists-and-tuples`


A name holds one value — but that value can itself be a **container** holding
many.

A **list** is written with square brackets, and holds its items in order:


In [ ]:
scores = [10, 20, 30]
print(scores)
print(len(scores))




`len(...)` asks the container how many items it holds. Asking is better than
remembering: the answer stays right when the list changes.

A list can hold values of different types, and its items can themselves be
lists. A list of lists is how a table (rows of equal length) is written before
any library gets involved:


In [ ]:
mixed = [1, "two", 3.0]
rows = [[1, 2, 3], [4, 5, 6]]

print(mixed)
print(rows)
print(len(rows), "rows of", len(rows[0]))




A **tuple** is written with parentheses instead. It holds items in order in
exactly the same way — the difference is that a tuple cannot be changed after it
is built:


In [ ]:
point = (3, 4)
print(point, len(point))




That is why a tuple is the usual way to hand back several answers at once: it is
a fixed group of values, not a collection you are still working on. Use a list
when the contents will grow or change, a tuple when they are one finished
answer.

The two convert into each other, and converting always produces a NEW container:


In [ ]:
values = [1, 2, 3]
as_tuple = tuple(values)
back = list(as_tuple)

print(as_tuple)
print(back)
print("same length:", len(values) == len(back))




A one-item tuple needs a trailing comma — `(5)` is just the number 5 with
brackets around it, while `(5,)` is a tuple holding it:


In [ ]:
print(type((5)).__name__)
print(type((5,)).__name__)


> **Watch out.** - **Brackets decide the type** — `[a, b]` is a list, `(a, b)` is a tuple. They
  hold the same values and are still different types.
- **`len` counts the OUTER level only** — `len([[1, 2, 3]])` is `1`: one inner
  list. What is inside it is a second question.
- **`(5)` is not a tuple** — a one-item tuple is `(5,)`.


A table as a list of lists, measured at both levels, then frozen into a tuple.


In [ ]:
rows = [[1, 2, 3], [4, 5, 6]]

n_rows = len(rows)
n_cols = len(rows[0])

print("rows:", n_rows)
print("columns:", n_cols)

shape = (n_rows, n_cols)
print("as one finished answer:", shape, type(shape).__name__)




Why each step:

1. `len(rows)` counts the inner lists — the rows of the table.
2. `len(rows[0])` reaches into the first row and counts what IT holds. Two
   levels, two separate measurements.
3. The pair is packed into a tuple because it is one finished answer with two
   parts, which is exactly the job a tuple is for.


<!-- dd:dd-q580 -->

### Problem 580 · faded — your turn

Build the list, then ask it how long it is.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([1, 2, 3], 3)
```


In [ ]:
def solve(a, b, c):
    """Build the list, then measure it."""
    items = [a, b, c]
    return (items, _____(items))


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (1, 2, 3)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(580)


In [ ]:
#@title 💡 Solution — Problem 580
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
def solve(a, b, c):
    """Build the list, then measure it."""
    items = [a, b, c]
    return (items, len(items))


example = (1, 2, 3)
print(solve(*example))


A table as a list of lists, measured at both levels, then frozen into a tuple.


In [ ]:
rows = [[1, 2, 3], [4, 5, 6]]

n_rows = len(rows)
n_cols = len(rows[0])

print("rows:", n_rows)
print("columns:", n_cols)

shape = (n_rows, n_cols)
print("as one finished answer:", shape, type(shape).__name__)




Why each step:

1. `len(rows)` counts the inner lists — the rows of the table.
2. `len(rows[0])` reaches into the first row and counts what IT holds. Two
   levels, two separate measurements.
3. The pair is packed into a tuple because it is one finished answer with two
   parts, which is exactly the job a tuple is for.


<!-- dd:dd-q581 -->

### Problem 581 · faded — your turn

Two values in a tuple — parentheses, not brackets.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(1, 2)
```


In [ ]:
def solve(a, b):
    """A tuple of the two values."""
    return _____a, b_____


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (1, 2)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(581)


In [ ]:
#@title 💡 Solution — Problem 581
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
def solve(a, b):
    """A tuple of the two values."""
    return (a, b)


example = (1, 2)
print(solve(*example))


<!-- dd:dd-q582 -->

### Problem 582 · independent

Write a function solve(rows) that takes a list whose items are themselves lists (all the same length) and returns the tuple (n_rows, n_cols): how many inner lists there are, and how many items the FIRST inner list holds.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(2, 3)
```


In [ ]:
def solve(rows):
    """Outer length, then inner length."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[1, 2, 3], [4, 5, 6]],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(582)


In [ ]:
#@title 💡 Solution — Problem 582
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
def solve(rows):
    """Outer length, then inner length."""
    return (len(rows), len(rows[0]))


example = ([[1, 2, 3], [4, 5, 6]],)
print(solve(*example))


<!-- dd:dd-q583 -->

### Problem 583 · independent

Write a function solve(items) that takes a list and returns the tuple (as_tuple, back_to_list, same_length): the list converted to a tuple, that tuple converted back to a list, and whether the two lengths agree.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
((1, 2, 3), [1, 2, 3], True)
```


In [ ]:
def solve(items):
    """A round trip between the two sequence types."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([1, 2, 3],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(583)


In [ ]:
#@title 💡 Solution — Problem 583
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
def solve(items):
    """A round trip between the two sequence types."""
    as_tuple = tuple(items)
    back = list(as_tuple)
    return (as_tuple, back, len(items) == len(back))


example = ([1, 2, 3],)
print(solve(*example))


#### Common mistakes

- **"A tuple is just a list that is written differently."** — It is a different
  type, and it cannot be changed once built. A grader comparing a list against a
  tuple reports them as unequal.
- **"`len` gives the total number of values."** — It gives the number of items at
  the top level. For a list of lists that is the number of rows.
- **"Converting a list to a tuple changes the list."** — It builds a new
  container. The original list is still there, unchanged.


<!-- dd:dd-kp-python-indexing -->

## Indexing — pulling one item out, counting from zero

`python.indexing`


Square brackets after a container mean **"give me the item at this position"**.
Positions are counted from **zero**, so the first item is at 0 and the last is at
`len(...) - 1`:


In [ ]:
letters = ["a", "b", "c"]

print(letters[0])
print(letters[1])
print(letters[2])
print("length is", len(letters), "so the last position is", len(letters) - 1)




Counting from zero is the single most common source of early off-by-one errors,
and it is worth saying out loud: `letters[1]` is the SECOND item.

Because "the last one" is needed constantly, negative positions count backwards
from the end. `-1` is the last item, `-2` the one before it:


In [ ]:
letters = ["a", "b", "c", "d"]

print(letters[-1])
print(letters[-2])




Negative indexing is not just shorter than `letters[len(letters) - 1]` — it is
safer, because there is no length to get wrong.

Reaching into nested data is **one index per level**, read left to right. The
first bracket picks the row; the second picks inside that row:


In [ ]:
rows = [[1, 2, 3], [4, 5, 6]]

print(rows[0])
print(rows[0][2])
print(rows[1][0])




`rows[0]` is itself a list, so `rows[0][2]` is "the third item of the first
row". The two indices are not interchangeable: `rows[2][0]` asks for a third row
that does not exist.

The two styles combine, which is how you name a corner of a table without
measuring anything:


In [ ]:
rows = [[1, 2, 3], [4, 5, 6]]

print("top-left    :", rows[0][0])
print("top-right   :", rows[0][-1])
print("bottom-right:", rows[-1][-1])




Asking for a position that does not exist stops the program with an
`IndexError` rather than guessing:


In [ ]:
letters = ["a", "b"]
try:
    print(letters[5])
except IndexError as exc:
    print("IndexError:", exc)


> **Watch out.** - **The first item is at 0** — so the last valid position is one LESS than the
  length. `items[len(items)]` is always one past the end.
- **`-1` is the last item, not "one before the start"** — the negative side
  starts at `-1`, there is no `-0`.
- **The first index is the row** — `rows[i][j]` picks row `i`, then item `j`
  inside it. Swapping them reads a different value, or fails.


One row of a table, pulled out and then read from both ends.


In [ ]:
rows = [[10, 20, 30], [40, 50]]

row = rows[1]

print("the row itself :", row)
print("how many items :", len(row))
print("its first item :", row[0])
print("its last item  :", row[-1])
print("not the same as the last ROW:", rows[-1])




Why each step:

1. `rows[1]` names the second row, so the reads below are all one level down and
   do not need a second bracket every time.
2. `row[0]` and `row[-1]` are the two ends of THAT row.
3. The last line is the trap in one line: `rows[-1]` is the last row of the
   whole table, which is a different depth from `row[-1]`.


<!-- dd:dd-q586 -->

### Problem 586 · faded — your turn

The first item lives at position zero.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
10
```


In [ ]:
def solve(items):
    """First item — position zero."""
    return items[_____]


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([10, 20, 30],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(586)


In [ ]:
#@title 💡 Solution — Problem 586
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
def solve(items):
    """First item — position zero."""
    return items[0]


example = ([10, 20, 30],)
print(solve(*example))


One row of a table, pulled out and then read from both ends.


In [ ]:
rows = [[10, 20, 30], [40, 50]]

row = rows[1]

print("the row itself :", row)
print("how many items :", len(row))
print("its first item :", row[0])
print("its last item  :", row[-1])
print("not the same as the last ROW:", rows[-1])




Why each step:

1. `rows[1]` names the second row, so the reads below are all one level down and
   do not need a second bracket every time.
2. `row[0]` and `row[-1]` are the two ends of THAT row.
3. The last line is the trap in one line: `rows[-1]` is the last row of the
   whole table, which is a different depth from `row[-1]`.


<!-- dd:dd-q587 -->

### Problem 587 · faded — your turn

The last item, counted back from the end rather than measured.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
30
```


In [ ]:
def solve(items):
    """Last item — count back from the end."""
    return items[_____]


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([10, 20, 30],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(587)


In [ ]:
#@title 💡 Solution — Problem 587
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
def solve(items):
    """Last item — count back from the end."""
    return items[-1]


example = ([10, 20, 30],)
print(solve(*example))


<!-- dd:dd-q588 -->

### Problem 588 · independent

Write a function solve(rows, i, j) that takes a list of inner lists and two positions, and returns the single value at row i, column j. Reading a nested structure is one index per level, left to right.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
2
```


In [ ]:
def solve(rows, i, j):
    """One index per level: pick the row, then the item."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[1, 2], [3, 4]], 0, 1)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(588)


In [ ]:
#@title 💡 Solution — Problem 588
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
def solve(rows, i, j):
    """One index per level: pick the row, then the item."""
    return rows[i][j]


example = ([[1, 2], [3, 4]], 0, 1)
print(solve(*example))


<!-- dd:dd-q589 -->

### Problem 589 · independent

Write a function solve(items) that returns the tuple (first, last, second_from_end) for a list of at least two items, using a negative position for the last two.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(1, 3, 2)
```


In [ ]:
def solve(items):
    """Front from zero, back from minus one."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([1, 2, 3],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(589)


In [ ]:
#@title 💡 Solution — Problem 589
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
def solve(items):
    """Front from zero, back from minus one."""
    return (items[0], items[-1], items[-2])


example = ([1, 2, 3],)
print(solve(*example))


#### Common mistakes

- **"`items[1]` is the first item."** — It is the second. Counting starts at 0.
- **"To get the last item I need the length."** — `items[-1]` does it with no
  measurement, and cannot go one past the end.
- **"`rows[i][j]` and `rows[j][i]` are the same."** — Only on a square table, and
  only by coincidence. The first index always picks the row.


<!-- dd:dd-kp-python-calling-functions -->

## Calling a function — arguments in, one value out

`python.calling-functions`


A **function** is a piece of work that has been given a name. **Calling** it is
writing that name followed by parentheses — and the parentheses are what make
the work happen:


In [ ]:
items = [10, 20, 30]

print(len(items))
print(len)




The second line prints something like `<built-in function len>`. Without
parentheses you are talking ABOUT the function; with them you are asking it to
run. Almost every "why did I get `<function ...>` back?" is this.

Values handed to a call are its **arguments**, separated by commas. Order
matters, because each position means something different to the function:


In [ ]:
print(round(3.14159, 2))
print(round(3.14159, 4))




`round` takes the number first and how many decimal places second. Swapping them
asks a different question — and usually an impossible one.

Every call produces **one value**, which you can name, print, or feed straight
into another call:


In [ ]:
numbers = [4, 1, 7]

print(sum(numbers))
print(max(numbers))
print(min(numbers))
print(sum(numbers) / len(numbers))




That last line is a call inside an expression built from two other calls: each
one is worked out first, and the results are then divided.

Some arguments are settings rather than data, and those are passed **by name**
so the call still reads as English. A named argument is written
`name=value` and can be left out entirely, in which case the function uses its
own default:


In [ ]:
items = [3, 1, 2]

print(sorted(items))
print(sorted(items, reverse=True))




`sorted` returns a NEW list and leaves the original alone — which you can check,
and which is why the name `sorted` is worth trusting:


In [ ]:
items = [3, 1, 2]
ordered = sorted(items)

print(ordered)
print(items)


> **Watch out.** - **No parentheses, no work** — `len` is the function, `len(items)` is the
  number.
- **Position is meaning** — `round(x, 2)` and `round(2, x)` are different
  questions.
- **A keyword argument must be named exactly** — `reverse=True` changes the
  direction; `True` on its own in that slot means something else entirely.


Three calls over one list, then a fourth built from two of them.


In [ ]:
readings = [4, 9, 2, 9]

count = len(readings)
total = sum(readings)
biggest = max(readings)

print("count  :", count)
print("total  :", total)
print("biggest:", biggest)
print("mean   :", round(total / count, 2))




Why each step:

1. `len`, `sum` and `max` each take the same list and each hand back one value.
   Naming those values is what lets the last line read as a formula.
2. The mean is `total / count` — two names, no recomputation.
3. `round(..., 2)` wraps that expression: the division happens first, and the
   rounded result is what gets printed.


<!-- dd:dd-q592 -->

### Problem 592 · faded — your turn

Ask the built-in how many items the list holds.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
3
```


In [ ]:
def solve(items):
    """Call len on the list."""
    return _____(items)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([1, 2, 3],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(592)


In [ ]:
#@title 💡 Solution — Problem 592
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
def solve(items):
    """Call len on the list."""
    return len(items)


example = ([1, 2, 3],)
print(solve(*example))


Three calls over one list, then a fourth built from two of them.


In [ ]:
readings = [4, 9, 2, 9]

count = len(readings)
total = sum(readings)
biggest = max(readings)

print("count  :", count)
print("total  :", total)
print("biggest:", biggest)
print("mean   :", round(total / count, 2))




Why each step:

1. `len`, `sum` and `max` each take the same list and each hand back one value.
   Naming those values is what lets the last line read as a formula.
2. The mean is `total / count` — two names, no recomputation.
3. `round(..., 2)` wraps that expression: the division happens first, and the
   rounded result is what gets printed.


<!-- dd:dd-q593 -->

### Problem 593 · faded — your turn

Two arguments, in the order the function expects them.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
3.14
```


In [ ]:
def solve(x, places):
    """round takes the value first, then how many places."""
    return round(x, _____)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (3.14159, 2)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(593)


In [ ]:
#@title 💡 Solution — Problem 593
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
def solve(x, places):
    """round takes the value first, then how many places."""
    return round(x, places)


example = (3.14159, 2)
print(solve(*example))


<!-- dd:dd-q594 -->

### Problem 594 · independent

Write a function solve(numbers) that returns the tuple (total, biggest, smallest) by calling the built-ins sum, max and min on the same list. Each call gives back ONE value.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(6, 3, 1)
```


In [ ]:
def solve(numbers):
    """Three built-in calls over one list."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([1, 2, 3],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(594)


In [ ]:
#@title 💡 Solution — Problem 594
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
def solve(numbers):
    """Three built-in calls over one list."""
    return (sum(numbers), max(numbers), min(numbers))


example = ([1, 2, 3],)
print(solve(*example))


<!-- dd:dd-q595 -->

### Problem 595 · independent

Write a function solve(items) that returns the list sorted from LARGEST to smallest, by calling sorted with the keyword argument reverse=True. A keyword argument names the setting it is changing.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[3, 2, 1]
```


In [ ]:
def solve(items):
    """sorted, with the direction named."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([3, 1, 2],)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(595)


In [ ]:
#@title 💡 Solution — Problem 595
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
def solve(items):
    """sorted, with the direction named."""
    return sorted(items, reverse=True)


example = ([3, 1, 2],)
print(solve(*example))


#### Common mistakes

- **"`len` gives the length."** — `len(items)` gives the length. `len` on its own
  is the function object, and returning it is the most common way a drill fails
  with no error message.
- **"Arguments can go in any order."** — Only keyword arguments can. Positional
  arguments mean whatever their position means.
- **"`sorted` sorts the list."** — It returns a new one. The original is
  unchanged, which is exactly why you have to keep the result.


<!-- dd:dd-kp-python-defining-functions -->

## Writing your own function — def and return

`python.defining-functions`


Every drill in this course asks for the same shape: **write a function called
`solve`**. This page is that shape.

`def` names a new function and lists the **parameters** it expects. The indented
lines below it are its **body** — the work — and `return` is how it hands a
value back to whoever called it:


In [ ]:
def double(n):
    """Twice n."""
    return n * 2


print(double(5))
print(double(2.5))




Three things to notice. The parameter `n` is a name that does not exist until
the function is called — it takes whatever value the caller passes. The
triple-quoted line is a **docstring**, a sentence saying what the function is
for. And the body is indented; the indentation is what says which lines belong
to the function.

`return` both hands the value back and **ends the call immediately**. Anything
after it never runs:


In [ ]:
def first_only(a, b):
    """The first argument, and proof that the second line is dead."""
    return a
    return b


print(first_only("kept", "never reached"))




A function with no `return` hands back `None`. This is the quietest bug in early
Python, because nothing goes wrong — you simply get nothing:


In [ ]:
def forgot(n):
    """Computes, and then throws it away."""
    n * 3


print(forgot(5))




To give more than one answer, return a tuple. One `return`, several values:


In [ ]:
def stats(a, b):
    """Sum and product, together."""
    return (a + b, a * b)


print(stats(2, 3))




A parameter can carry a **default**, used only when the caller says nothing.
That is what makes one function serve two calls:


In [ ]:
def scale(x, times=2):
    """Multiply x, doubling unless told otherwise."""
    return x * times


print(scale(5))
print(scale(5, 10))




Functions can also be defined INSIDE other functions, which is how a repeated
step gets a name without leaking out into the rest of the program:


In [ ]:
def ends_doubled(values):
    """Double the first and last items, via one local helper."""
    def double(n):
        """Twice n."""
        return n * 2

    return (double(values[0]), double(values[-1]))


print(ends_doubled([1, 2, 7]))


> **Watch out.** - **No `return` means `None`** — the work happens and the answer is discarded.
  A grader sees `None` and reports a failure with nothing obviously wrong.
- **`return` stops the function** — a second `return` on the next line is dead
  code, not a second answer. Return a tuple instead.
- **A default belongs in the `def` line** — `def solve(x, times=2)`. Writing
  `times = 2` in the body ignores whatever the caller passed.


One function, two parameters, a default, and a tuple of two answers.


In [ ]:
def summarize(values, places=2):
    """Return (count, mean) with the mean rounded to `places` places."""
    count = len(values)
    mean = round(sum(values) / count, places)
    return (count, mean)


print(summarize([1, 2, 4]))
print(summarize([1, 2, 4], 0))




Why each step:

1. The `def` line is the contract: two parameters, the second optional.
2. `count` and `mean` are named inside the body. Those names exist only while
   the call is running.
3. One `return` hands back both answers as a tuple — and the two printed lines
   show the default being used, then overridden.


<!-- dd:dd-q598 -->

### Problem 598 · faded — your turn

The work is done; `return` is what hands it back.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
6
```


In [ ]:
def solve(x):
    """Hand back three times x."""
    _____ x * 3


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (2,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(598)


In [ ]:
#@title 💡 Solution — Problem 598
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
def solve(x):
    """Hand back three times x."""
    return x * 3


example = (2,)
print(solve(*example))


One function, two parameters, a default, and a tuple of two answers.


In [ ]:
def summarize(values, places=2):
    """Return (count, mean) with the mean rounded to `places` places."""
    count = len(values)
    mean = round(sum(values) / count, places)
    return (count, mean)


print(summarize([1, 2, 4]))
print(summarize([1, 2, 4], 0))




Why each step:

1. The `def` line is the contract: two parameters, the second optional.
2. `count` and `mean` are named inside the body. Those names exist only while
   the call is running.
3. One `return` hands back both answers as a tuple — and the two printed lines
   show the default being used, then overridden.


<!-- dd:dd-q599 -->

### Problem 599 · faded — your turn

Two parameters, and both of them have to appear in the body.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
6
```


In [ ]:
def solve(a, b):
    """Two parameters, one returned value."""
    return a * _____


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (2, 3)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(599)


In [ ]:
#@title 💡 Solution — Problem 599
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
def solve(a, b):
    """Two parameters, one returned value."""
    return a * b


example = (2, 3)
print(solve(*example))


<!-- dd:dd-q600 -->

### Problem 600 · independent

Write a function solve(x, times=2) whose second parameter has a DEFAULT of 2, and which returns x multiplied by `times`. Callers that pass one argument must get double; callers that pass two must get their own multiplier.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
10
```


In [ ]:
def solve(x, times=2):
    """A default is used only when the caller says nothing."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (5,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(600)


In [ ]:
#@title 💡 Solution — Problem 600
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
def solve(x, times=2):
    """A default is used only when the caller says nothing."""
    return x * times


example = (5,)
print(solve(*example))


<!-- dd:dd-q601 -->

### Problem 601 · independent

Write a function solve(a, b) that returns TWO values as a tuple: their sum and their product. One `return` can hand back a tuple, which is how a function gives more than one answer.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(5, 6)
```


In [ ]:
def solve(a, b):
    """Two answers, one return."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (2, 3)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(601)


In [ ]:
#@title 💡 Solution — Problem 601
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
def solve(a, b):
    """Two answers, one return."""
    return (a + b, a * b)


example = (2, 3)
print(solve(*example))


#### Common mistakes

- **"The function ran, so it returned something."** — Only `return` returns.
  Without it every call hands back `None`.
- **"Two returns give two answers."** — The first one ends the call. A tuple is
  how one return carries several values.
- **"Parameters are variables I set."** — They are named by the `def` line and
  filled in by the caller, fresh on every call.


<!-- dd:dd-kp-python-dots-and-imports -->

## Dots — importing a library, attributes, and methods

`python.dots-and-imports`


Most of the code you will write reaches into something else with a **dot**. Three
different things use that same dot, and telling them apart is what this page is
for.

**A module.** `import` brings in a library; the dot then reaches inside it for
the things it holds:


In [ ]:
import math

print(math.sqrt(9))
print(math.floor(2.7), math.ceil(2.7))




`import math as m` gives the same module a shorter name. This is not cosmetic —
it is the convention the rest of this course runs on: the tensor library you
meet in the next lesson is always brought in under a two-letter alias, and every
line of it after that is written through the dot.


In [ ]:
import math as m

print(m.sqrt(16))
print(m.floor(-1.2))




**An attribute** is a value that belongs to something. It is read with a dot and
**no parentheses**, because there is no work to do — the value is already there:


In [ ]:
import math

print(math.pi)




**A method** is a function that belongs to something. It is reached with a dot
and, being a function, still has to be **called**:


In [ ]:
text = "hello"

print(text.upper())
print(text)




Notice `text` is unchanged: `upper()` returned a NEW string. Notice too that the
call is `text.upper()` — the string it works on is the thing before the dot, so
nothing goes in the parentheses.

That is the distinction to hold on to, because it decides whether parentheses
belong:

| Written | Means |
|---|---|
| `math.pi` | an attribute — a value, read as-is |
| `math.sqrt` | the function itself, not run |
| `math.sqrt(9)` | the call — the number 3.0 |
| `text.upper` | the method itself, not run |
| `text.upper()` | the call — the capitalised text |

Not everything is a method. `len` is a plain built-in function that takes the
value as an argument, and strings have no `.len()` at all:


In [ ]:
text = "hello"

print(text.upper())
print(len(text))




Some methods change the thing they belong to and return `None`. `list.append` is
the one you meet first, and the `None` is what surprises people:


In [ ]:
items = [1, 2]
result = items.append(3)

print(items)
print("append returned:", result)




So append on its own line, then hand back the LIST — never the result of the
append.


> **Watch out.** - **Attribute or call?** — `math.pi` has no parentheses because it is a value.
  `math.sqrt(9)` has them because it is work.
- **A method with no parentheses is not run** — returning `text.upper` hands back
  a method object, and nothing complains.
- **`.append()` returns `None`** — it changes the list in place. Returning what
  it gave you returns nothing.


The three kinds of dot in one place, on the same two values.


In [ ]:
import math

text = "delta"
x = 6.25

shouted = text.upper()
length = len(text)
root = math.sqrt(x)

print("method on the value  :", shouted)
print("built-in on the value:", length)
print("function in a module :", root)
print("attribute, no call   :", math.pi)




Why each step:

1. `text.upper()` is a method: it belongs to the string, and the parentheses run
   it.
2. `len(text)` is a built-in function: the string goes INSIDE the parentheses.
   Same job shape, opposite arrangement.
3. `math.sqrt(x)` reaches into the module, then calls what it found.
4. `math.pi` is read with no parentheses at all, because it is already a value.


<!-- dd:dd-q604 -->

### Problem 604 · faded — your turn

Reach into the module with a dot, then call what you found.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
3.0
```


In [ ]:
import math


def solve(x):
    """Reach into the module with a dot, then call."""
    return math._____(x)


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (9,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(604)


In [ ]:
#@title 💡 Solution — Problem 604
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import math


def solve(x):
    """Reach into the module with a dot, then call."""
    return math.sqrt(x)


example = (9,)
print(solve(*example))


The three kinds of dot in one place, on the same two values.


In [ ]:
import math

text = "delta"
x = 6.25

shouted = text.upper()
length = len(text)
root = math.sqrt(x)

print("method on the value  :", shouted)
print("built-in on the value:", length)
print("function in a module :", root)
print("attribute, no call   :", math.pi)




Why each step:

1. `text.upper()` is a method: it belongs to the string, and the parentheses run
   it.
2. `len(text)` is a built-in function: the string goes INSIDE the parentheses.
   Same job shape, opposite arrangement.
3. `math.sqrt(x)` reaches into the module, then calls what it found.
4. `math.pi` is read with no parentheses at all, because it is already a value.


<!-- dd:dd-q605 -->

### Problem 605 · faded — your turn

A method belongs to the value — and still has to be called.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
'HI'
```


In [ ]:
def solve(text):
    """A method belongs to the value, and still has to be called."""
    return text.upper_____


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ('hi',)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(605)


In [ ]:
#@title 💡 Solution — Problem 605
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
def solve(text):
    """A method belongs to the value, and still has to be called."""
    return text.upper()


example = ('hi',)
print(solve(*example))


<!-- dd:dd-q606 -->

### Problem 606 · independent

Write a function solve(text) that returns the tuple (shouted, size): the string in capitals, and how long it is. One of those is a METHOD on the string, the other is a built-in function that takes the string as an argument.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
('HI', 2)
```


In [ ]:
def solve(text):
    """A method on the value, and a function taking the value."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ('hi',)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(606)


In [ ]:
#@title 💡 Solution — Problem 606
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
def solve(text):
    """A method on the value, and a function taking the value."""
    return (text.upper(), len(text))


example = ('hi',)
print(solve(*example))


<!-- dd:dd-q607 -->

### Problem 607 · independent

Write a function solve(x) that imports `math` under the shorter name `m` and returns the tuple (floor_value, ceil_value): x rounded down, and x rounded up. `import math as m` gives the same module a shorter name.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(2, 3)
```


In [ ]:
import math as m



def solve(x):
    """The same module, under a shorter name."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (2.3,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(607)


In [ ]:
#@title 💡 Solution — Problem 607
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import math as m


def solve(x):
    """The same module, under a shorter name."""
    return (m.floor(x), m.ceil(x))


example = (2.3,)
print(solve(*example))


#### Common mistakes

- **"Everything after a dot needs parentheses."** — Attributes do not.
  `math.pi()` is an error; `math.pi` is a number.
- **"`text.len()` should work."** — Length is a built-in function, not a string
  method. The arrangement is `len(text)`.
- **"`items.append(x)` gives me the longer list."** — It gives `None` and changes
  `items` in place.
